In [ ]:
import os
import nltk
from nltk import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from whoosh.index import create_in
from whoosh.fields import *
from whoosh.analysis import StandardAnalyzer
from whoosh import qparser
from whoosh import scoring
import whoosh.index as index

import pytrec_eval
import math


nltk.download('punkt_tab')
nltk.download('stopwords')
stoplist = stopwords.words("english")
stoplist.append('oh')
puncts = ['.', ',', ':', '`', '"', "'", '!', '?', "``", "''"]
ps = PorterStemmer()

In [ ]:
def preprocess(tok, stemmer=ps, punctlist=puncts, stopwords=stoplist):
  tok = tok.lower()
  if tok.isdigit():
    return None
  if tok.isnumeric():
    return None
  if tok in punctlist:
    return None
  if tok in stopwords:
    return None
  return stemmer.stem(tok)

def indexing(src, idx="ind"):
  if src[-1] != '/':
    src += '/'
  schema = Schema(docid=STORED(), content=TEXT(stored=True, analyzer=StandardAnalyzer()))
  ix = create_in(idx, schema)
  writer = ix.writer()

  files = os.listdir(src)
  for f in files:
    r = open(src + f, encoding="cp1252")
    terms = []
    for s in r:
      for sent in sent_tokenize(s.strip()):
        for tok in word_tokenize(sent):
          tok = preprocess(tok)
          if tok != None:
            terms.append(tok)
    r.close()
    cont = " ".join(terms)
    writer.add_document(docid="{}".format(f.split(".")[0]), content=cont)
  writer.commit()

In [ ]:
indexing("Cranfield/Cranfield", "ind")

In [ ]:
def readGroundTruth(src):
  if src[-1] != '/':
    src += '/'

  GT = {}
  for f in os.listdir(src):
    r = open(src + f)
    rel = {}
    for s in r:
      s = s.strip()
      sp = s.split("\t")
      if len(sp) < 2:
        continue
      did = sp[0].split(" ")[1]
      rel[did] = int(sp[1])
    GT[f.split(".")[0]] = rel
    r.close()
  return GT

In [ ]:
GroundTruth = readGroundTruth("Cranfield/RES")
print(GroundTruth)

In [ ]:
def readQuery(src):
  qry = {}
  r = open(src)
  for s in r:
    s = s.strip()
    ps = s.split("\t")
    terms = []
    for sent in sent_tokenize(ps[1]):
      for tok in word_tokenize(sent):
        tok = preprocess(tok)
        if tok != None:
          terms.append(tok)
    qry[ps[0]] = " ".join(terms)
  return qry

In [ ]:
Queries = readQuery("Cranfield/query.txt")
print(Queries)

In [ ]:
def processQueries(ind, qry):
    idx = index.open_dir(ind)
    weighting = scoring.BM25F()  # default BM25F

    searcher = idx.searcher(weighting=weighting)
    parser = qparser.QueryParser("content", idx.schema, group=qparser.OrGroup)

    RET = {}
    for key in qry:
        if method != "vector":
            query = parser.parse(qry[key])
            results = searcher.search(query, limit=None)
            RET[key] = {r["docid"]: r.score for r in results}
        else:
            query_emb = model.encode(qry[key])
            RET[key] = {docid: float(util.cos_sim(query_emb, emb)) for docid, emb in doc_embeddings.items()}

    return RET


### 2. Chia query test

In [ ]:
import random

def split_queries(qry, ratio=0.1, seed=42):
    keys = list(qry.keys())
    random.Random(seed).shuffle(keys)
    cut = int(len(keys) * ratio)
    tune_keys = keys[:cut]
    eval_keys = keys[cut:]

    Q_tune = {k: qry[k] for k in tune_keys}
    Q_eval = {k: qry[k] for k in eval_keys}

    return Q_tune, Q_eval


### 3. Pseudo relevance feedback (giả định relevance)

In [ ]:
def pseudo_relevance(results, K=10):
    rel = {}
    for i, r in enumerate(results[:K]):
        rel[r["docid"]] = 1   # relevant
    return rel


### 4. Hàm chạy BM25 với (k, b) tùy chỉnh

In [ ]:
from whoosh import index, scoring, qparser

def processQueries_BM25(ind, qry, k1=1.2, b=0.75):
    idx = index.open_dir(ind)
    searcher = idx.searcher(
        weighting=scoring.BM25F(k1=k1, B=b)
    )
    parser = qparser.QueryParser("content", idx.schema, group=qparser.OrGroup)

    RET = {}
    for qid in qry:
        query = parser.parse(qry[qid])
        results = searcher.search(query, limit=None)

        rel = {}
        for r in results:
            rel[r["docid"]] = r.score
        RET[qid] = rel

    return RET


### 5. Tuning tham số k, b bằng PRF

In [ ]:
import numpy as np
import pytrec_eval

def tune_bm25(ind, Q_tune, groundtruth):
    k_values = [0.8, 1.0, 1.2, 1.5, 2.0]
    b_values = [0.3, 0.5, 0.75, 0.9]

    best_score = -1
    best_params = None

    evaluator = pytrec_eval.RelevanceEvaluator(
        groundtruth, {"map"}
    )

    for k in k_values:
        for b in b_values:
            run = processQueries_BM25(ind, Q_tune, k1=k, b=b)
            scores = evaluator.evaluate(run)

            MAP = np.mean([scores[q]["map"] for q in scores])

            print(f"k={k}, b={b}, MAP={MAP:.4f}")

            if MAP > best_score:
                best_score = MAP
                best_params = (k, b)

    return best_params, best_score


### 6. Đánh giá trên 50% còn lại

In [ ]:
def evaluate_final(ind, Q_eval, groundtruth, k, b):
    run = processQueries_BM25(ind, Q_eval, k1=k, b=b)

    evaluator = pytrec_eval.RelevanceEvaluator(
        groundtruth, {"map", "ndcg", "recip_rank"}
    )

    scores = evaluator.evaluate(run)

    metrics = {
        "MAP": np.mean([scores[q]["map"] for q in scores]),
        "nDCG": np.mean([scores[q]["ndcg"] for q in scores]),
        "MRR": np.mean([scores[q]["recip_rank"] for q in scores])
    }

    return metrics


### 7. Quy trình hoàn chỉnh (pipeline)

In [ ]:
Q_tune, Q_eval = split_queries(qry)

(best_k, best_b), best_map = tune_bm25(
    index_dir,
    Q_tune,
    groundtruth_tune
)

print("Best params:", best_k, best_b)

final_metrics = evaluate_final(
    index_dir,
    Q_eval,
    groundtruth_eval,
    best_k,
    best_b
)

print(final_metrics)
